# PySpark + MinIO — Starter Notebook

This notebook verifies that PySpark can connect to the Spark master
and read/write data to MinIO (S3-compatible storage).

In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MinIO-Test") \
    .master(os.environ.get("SPARK_MASTER", "local[*]")) \
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "localhost")) \
    .config("spark.driver.port", os.environ.get("SPARK_DRIVER_PORT", "45083")) \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "512m") \
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"]) \
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ACCESS_KEY"]) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_SECRET_KEY"]) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
# Create a test DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("product", StringType(), False),
    StructField("region", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("price", DoubleType(), False)
])

data = [
    ("Widget A", "North", 100, 9.99),
    ("Widget A", "South", 150, 9.99),
    ("Widget B", "North", 200, 14.50),
    ("Widget B", "South", 80, 14.50),
    ("Widget C", "North", 50, 24.99),
    ("Widget C", "South", 300, 24.99),
]

df = spark.createDataFrame(data, schema)
df.show()

In [ ]:
# Write to MinIO as Parquet (raw-data bucket)
df.write \
    .mode("overwrite") \
    .parquet("s3a://raw-data/test/sales.parquet")

print("Written to s3a://raw-data/test/sales.parquet")

In [ ]:
# Read it back from MinIO
df_read = spark.read.parquet("s3a://raw-data/test/sales.parquet")
df_read.show()
print(f"Row count: {df_read.count()}")

In [ ]:
# Aggregation — revenue by product
from pyspark.sql import functions as F

revenue = df_read \
    .withColumn("revenue", F.col("quantity") * F.col("price")) \
    .groupBy("product") \
    .agg(
        F.sum("quantity").alias("total_qty"),
        F.sum("revenue").alias("total_revenue"),
        F.avg("price").alias("avg_price")
    ) \
    .orderBy(F.desc("total_revenue"))

revenue.show()

# Write aggregated results to processed-data bucket (gold layer)
revenue.write \
    .mode("overwrite") \
    .parquet("s3a://processed-data/gold/revenue_by_product.parquet")

print("Written to s3a://processed-data/gold/revenue_by_product.parquet")

In [ ]:
spark.stop()
print("Done! Spark session closed.")